<a href="https://colab.research.google.com/github/swathigowroju/AI-Full-Stack/blob/main/Week1_AI%20Fullstack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Python Refresh + Your First LLM Call

**AI Full Stack / LLM Engineering Program**

This notebook is the hands-on companion to the Week 1 slides. It has two parts:

1. **Python Refresh** — quick exercises on functions, loops, and dictionaries.
2. **Your First LLM Call** — using the free-tier Groq API to call `llama-3.3-70b-versatile`, then experimenting with tokens, system prompts, and temperature.

> **Before you start:** In Colab, click the key icon 🔑 in the left sidebar → **Secrets** → add a secret named exactly `GROQ_API_KEY` with your Groq API key as the value → toggle **Notebook access** on. Get a free key at https://console.groq.com/keys.

## Part 1 — Python Refresh

### 1.1 Functions

A function wraps a piece of logic so you can reuse it. Every LLM helper we write this semester will be a function that takes a prompt in and returns a reply.

In [ ]:
def shout(text):
    """Return the text in uppercase with an exclamation mark."""
    return text.upper() + "!"

print(shout("hello groq"))

**Try it yourself:** write a function `count_words(text)` that returns the number of words in `text`.

In [ ]:
def count_words(text):
    # TODO: split the text on whitespace and return the length
    pass

# count_words("this should return five words")

### 1.2 Loops

Loops let us send several prompts without repeating code — useful for batch questions or retries.

In [ ]:
questions = [
    "What is a variable?",
    "What is a loop?",
    "What is a dictionary?",
]

for i, q in enumerate(questions, start=1):
    print(f"{i}. {q}")

### 1.3 Dictionaries

LLM APIs represent every message as a dictionary with `role` and `content` keys. Getting comfortable with dictionaries now makes the API code below feel familiar.

In [ ]:
message = {
    "role": "user",
    "content": "Explain recursion in one sentence."
}

print(message["role"])
print(message["content"])

**Try it yourself:** build a list of two message dictionaries — one with `role: "system"` and one with `role: "user"` — like a real chat request will need below.

In [ ]:
conversation = [
    # {"role": "system", "content": "..."},
    # {"role": "user", "content": "..."},
]

conversation

## Part 2 — Your First LLM Call

### 2.1 Install the Groq SDK

In [ ]:
!pip install -q groq

### 2.2 Load the API key safely from Colab Secrets

We never type the key into a cell. `userdata.get("GROQ_API_KEY")` reads it from Colab's encrypted secret store at runtime.

In [ ]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Key loaded:", "Yes" if os.environ.get("GROQ_API_KEY") else "No")

### 2.3 Make the client and your first call

We use `llama-3.3-70b-versatile` — the free-tier model this course standardizes on.

In [ ]:
from google.colab import userdata
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
#llama-3.3-70b-versatile
MODEL = "openai/gpt-oss-120b"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "In one sentence, what is a large language model?"}
    ],
)

print(response.choices[0].message.content)

### 2.4 Wrap it in a reusable function

This is the function-plus-dictionary pattern from Part 1, now doing real work.

In [ ]:
def ask_llm(user_prompt, system_prompt=None, temperature=0.7, model=MODEL):
    """Send a single-turn chat request to Groq and return the reply text."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response

result = ask_llm("Give me a fun fact about Python the language.")
#ask_llm.__doc__
print(result.choices[0].message.content)

### 2.5 Experiment: Looking at token usage

Every response includes a `usage` object — this is how you see the tokens from the slides made concrete.

In [ ]:
result = ask_llm("Explain what a token is, in exactly two sentences.")

print("Reply:\n", result.choices[0].message.content)
print("\nToken usage:")
print("  Prompt tokens:", result.usage.prompt_tokens)
print("  Completion tokens:", result.usage.completion_tokens)
print("  Total tokens:", result.usage.total_tokens)

**Try it yourself:** ask a much longer question and compare `prompt_tokens` to the number of words you typed. Is it more or fewer tokens than words?

In [ ]:
# your longer question here


### 2.6 Experiment: System prompt vs. no system prompt

Same user question, with and without a system prompt shaping the persona.

In [ ]:
question = "Should I use a list or a dictionary to store student names and marks?"

no_system = ask_llm(question)
print("--- Without a system prompt ---")
print(no_system.choices[0].message.content)

print("\n" + "="*60 + "\n")

with_system = ask_llm(
    question,
    system_prompt="You are a concise coding tutor for first-year B.Tech students. Answer in at most 3 short bullet points.",
)
print("--- With a system prompt ---")
print(with_system.choices[0].message.content)

### 2.7 Experiment: Temperature

Same prompt, three temperatures — the experiment previewed in the slides. Run this a couple of times and notice how much (or little) the `1.4` output changes between runs compared to `0.0`.

In [ ]:
prompt = "Write a one-line tagline for a college AI club."

for temp in [0.0, 0.7, 1.4]:
    result = ask_llm(prompt, temperature=temp)
    print(f"Temperature {temp}:")
    print(" ", result.choices[0].message.content)
    print()

**Try it yourself:** pick your own prompt and re-run the temperature experiment above. Which temperature gave the most useful result for *your* prompt?

In [ ]:
# your own prompt + temperature experiment here


## Deliverable

Turn the cells above into a single, clean script (e.g. `week1_llm_call.py`) that:

1. Loads `GROQ_API_KEY` safely (from an environment variable, not a hardcoded string).
2. Defines an `ask_llm(...)` function.
3. Calls it with one prompt and prints the answer.

Then push the script to your GitHub repository. **Do not commit your API key** — double-check the file before pushing.